# 10 - Prediction Only (Daily, Weekly, Monthly, Quarterly, Yearly)

This notebook is dedicated to **forward prediction only** with detailed outputs.

It provides no-leakage forecasts for these assets:
- S&P 500
- Nasdaq
- Gold
- Silver

For each asset, it predicts:
- Daily direction
- Weekly direction
- Monthly direction
- Quarterly direction
- Yearly direction

Label convention: `1 = UP (Close > Open)`, `0 = DOWN/FLAT`.

In [ ]:
import os
import sys
from datetime import datetime, timedelta

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))
import config

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

try:
    import yfinance as yf
except ModuleNotFoundError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'yfinance', '-q'])
    import yfinance as yf

np.random.seed(config.RANDOM_STATE)

In [ ]:
# High-accuracy intraday feature helper
def add_intraday_features(df):
    d = df.copy()
    d['Range'] = (d['High'] - d['Low']) / d['Open']
    d['High_to_Open'] = (d['High'] - d['Open']) / d['Open']
    d['Low_to_Open'] = (d['Low'] - d['Open']) / d['Open']
    d['Mid_to_Open'] = ((d['High'] + d['Low']) / 2 - d['Open']) / d['Open']
    d['Open_to_prev_close'] = (d['Open'] - d['Close'].shift(1)) / d['Close'].shift(1)
    return d


def _fit_model_tuned(X_train, y_train):
    model = RandomForestClassifier(
        n_estimators=1000,
        max_depth=8,
        min_samples_leaf=1,
        class_weight='balanced',
        random_state=config.RANDOM_STATE,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    return model


INTRADAY_FEATURES = ['High_to_Open', 'Low_to_Open', 'Open_to_prev_close', 'Range', 'Mid_to_Open']

ASSETS = {
    'S&P 500': '^GSPC',
    'Nasdaq': '^IXIC',
    'Gold': 'GC=F',
    'Silver': 'SI=F',
}


def predict_daily(raw_daily, today):
    feat = add_intraday_features(raw_daily)
    feat = feat.dropna()

    X = feat[INTRADAY_FEATURES]
    y = feat['Direction']

    train_mask = X.index < today
    X_train = X.loc[train_mask]
    y_train = y.loc[train_mask].astype(int)
    if len(X_train) < 200:
        raise ValueError(f'Not enough daily training rows: {len(X_train)}')

    model = _fit_model_tuned(X_train.values, y_train.values)

    pred_date = today if today in X.index else X.index.max()
    x_now = X.loc[[pred_date]]
    if x_now.isna().any(axis=1).iloc[0]:
        raise ValueError(f'Daily feature row for {pred_date.date()} contains NaN values.')

    pred_label = int(model.predict(x_now.values)[0])
    pred_proba_up = float(model.predict_proba(x_now.values)[0, 1])

    return {
        'prediction_date': pred_date,
        'input_date': pred_date - pd.Timedelta(days=1),
        'pred_label': pred_label,
        'pred_proba_up': pred_proba_up,
        'open_price': float(raw_daily.loc[pred_date, 'Open']),
        'latest_price': float(raw_daily.loc[pred_date, 'Close']),
        'in_progress': pred_date.date() == today.date(),
    }


def predict_weekly(raw_daily, today):
    weekly = raw_daily.resample('W-FRI').agg({
        'Open': 'first',
        'High': 'max',
        'Low': 'min',
        'Close': 'last',
        'Volume': 'sum',
    }).dropna(subset=['Open', 'Close'])
    
    feat = add_intraday_features(weekly)
    feat = feat.dropna()

    X = feat[INTRADAY_FEATURES]
    y = (feat['Close'] > feat['Open']).astype(int)

    days_to_friday = (4 - today.weekday()) % 7
    current_week_end = today + pd.Timedelta(days=days_to_friday)

    train_mask = X.index < current_week_end
    X_train = X.loc[train_mask]
    y_train = y.loc[train_mask]
    if len(X_train) < 60:
        raise ValueError(f'Not enough weekly training rows: {len(X_train)}')

    model = _fit_model_tuned(X_train.values, y_train.values)

    pred_date = current_week_end if current_week_end in X.index else X.index.max()
    x_now = X.loc[[pred_date]]
    if x_now.isna().any(axis=1).iloc[0]:
        raise ValueError(f'Weekly feature row for {pred_date.date()} contains NaN values.')

    pred_label = int(model.predict(x_now.values)[0])
    pred_proba_up = float(model.predict_proba(x_now.values)[0, 1])

    return {
        'prediction_date': pred_date,
        'input_date': pred_date - pd.Timedelta(days=7),
        'pred_label': pred_label,
        'pred_proba_up': pred_proba_up,
        'open_price': float(weekly.loc[pred_date, 'Open']),
        'latest_price': float(weekly.loc[pred_date, 'Close']),
        'in_progress': pred_date >= today,
    }


def predict_monthly(raw_daily, today):
    monthly = raw_daily.resample('ME').agg({
        'Open': 'first',
        'High': 'max',
        'Low': 'min',
        'Close': 'last',
        'Volume': 'sum',
    }).dropna(subset=['Open', 'Close'])
    
    feat = add_intraday_features(monthly)
    feat = feat.dropna()

    X = feat[INTRADAY_FEATURES]
    y = (feat['Close'] > feat['Open']).astype(int)

    current_month_end = today.to_period('M').to_timestamp('M')

    train_mask = X.index < current_month_end
    X_train = X.loc[train_mask]
    y_train = y.loc[train_mask]
    if len(X_train) < 24:
        raise ValueError(f'Not enough monthly training rows: {len(X_train)}')

    model = _fit_model_tuned(X_train.values, y_train.values)

    pred_date = current_month_end if current_month_end in X.index else X.index.max()
    x_now = X.loc[[pred_date]]
    if x_now.isna().any(axis=1).iloc[0]:
        raise ValueError(f'Monthly feature row for {pred_date.date()} contains NaN values.')

    pred_label = int(model.predict(x_now.values)[0])
    pred_proba_up = float(model.predict_proba(x_now.values)[0, 1])

    prev_month_end = (pred_date.to_period('M') - 1).to_timestamp('M')

    return {
        'prediction_date': pred_date,
        'input_date': prev_month_end,
        'pred_label': pred_label,
        'pred_proba_up': pred_proba_up,
        'open_price': float(monthly.loc[pred_date, 'Open']),
        'latest_price': float(monthly.loc[pred_date, 'Close']),
        'in_progress': pred_date >= current_month_end,
    }


def predict_quarterly(raw_daily, today):
    quarterly = raw_daily.resample('QE').agg({
        'Open': 'first',
        'High': 'max',
        'Low': 'min',
        'Close': 'last',
        'Volume': 'sum',
    }).dropna(subset=['Open', 'Close'])
    
    feat = add_intraday_features(quarterly)
    feat = feat.dropna()

    X = feat[INTRADAY_FEATURES]
    y = (feat['Close'] > feat['Open']).astype(int)

    current_quarter_end = today.to_period('Q').to_timestamp('Q')

    train_mask = X.index < current_quarter_end
    X_train = X.loc[train_mask]
    y_train = y.loc[train_mask]
    if len(X_train) < 40:
        raise ValueError(f'Not enough quarterly training rows: {len(X_train)}')

    model = _fit_model_tuned(X_train.values, y_train.values)

    pred_date = current_quarter_end if current_quarter_end in X.index else X.index.max()
    x_now = X.loc[[pred_date]]
    if x_now.isna().any(axis=1).iloc[0]:
        raise ValueError(f'Quarterly feature row for {pred_date.date()} contains NaN values.')

    pred_label = int(model.predict(x_now.values)[0])
    pred_proba_up = float(model.predict_proba(x_now.values)[0, 1])

    prev_quarter_end = (pred_date.to_period('Q') - 1).to_timestamp('Q')

    return {
        'prediction_date': pred_date,
        'input_date': prev_quarter_end,
        'pred_label': pred_label,
        'pred_proba_up': pred_proba_up,
        'open_price': float(quarterly.loc[pred_date, 'Open']),
        'latest_price': float(quarterly.loc[pred_date, 'Close']),
        'in_progress': pred_date >= current_quarter_end,
    }


def predict_yearly(raw_daily, today):
    yearly = raw_daily.resample('YE').agg({
        'Open': 'first',
        'High': 'max',
        'Low': 'min',
        'Close': 'last',
        'Volume': 'sum',
    }).dropna(subset=['Open', 'Close'])
    
    feat = add_intraday_features(yearly)
    feat = feat.dropna()

    X = feat[INTRADAY_FEATURES]
    y = (feat['Close'] > feat['Open']).astype(int)

    current_year_end = today.to_period('Y').to_timestamp('Y')

    train_mask = X.index < current_year_end
    X_train = X.loc[train_mask]
    y_train = y.loc[train_mask]
    if len(X_train) < 10:
        raise ValueError(f'Not enough yearly training rows: {len(X_train)}')

    model = _fit_model_tuned(X_train.values, y_train.values)

    pred_date = current_year_end if current_year_end in X.index else X.index.max()
    x_now = X.loc[[pred_date]]
    if x_now.isna().any(axis=1).iloc[0]:
        raise ValueError(f'Yearly feature row for {pred_date.date()} contains NaN values.')

    pred_label = int(model.predict(x_now.values)[0])
    pred_proba_up = float(model.predict_proba(x_now.values)[0, 1])

    prev_year_end = (pred_date.to_period('Y') - 1).to_timestamp('Y')

    return {
        'prediction_date': pred_date,
        'input_date': prev_year_end,
        'pred_label': pred_label,
        'pred_proba_up': pred_proba_up,
        'open_price': float(yearly.loc[pred_date, 'Open']),
        'latest_price': float(yearly.loc[pred_date, 'Close']),
        'in_progress': pred_date >= current_year_end,
    }

In [3]:
today = pd.Timestamp(datetime.now().date())
fetch_end = (today + timedelta(days=1)).strftime('%Y-%m-%d')

market_data = {}
for asset_name, ticker in ASSETS.items():
    raw = yf.download(
        ticker,
        start=config.START_DATE,
        end=fetch_end,
        auto_adjust=True,
        progress=False,
    )

    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    if raw.empty:
        print(f'{asset_name:8s} ({ticker}) -> no data returned, skipping.')
        continue

    raw = raw.rename_axis('Date').sort_index()
    raw = raw[['Open', 'High', 'Low', 'Close', 'Volume']].dropna(subset=['Open', 'Close'])
    market_data[asset_name] = raw
    print(f'{asset_name:8s} ({ticker}) -> {len(raw):,} rows')

if not market_data:
    raise ValueError('No asset data was downloaded from yfinance.')

S&P 500  (^GSPC) -> 4,088 rows
Nasdaq   (^IXIC) -> 4,088 rows
Gold     (GC=F) -> 4,087 rows
Silver   (SI=F) -> 4,087 rows


## Daily Prediction

In [4]:
for asset_name, raw_daily in market_data.items():
    print('=' * 70)
    print(f'Asset                     : {asset_name}')

    try:
        out = predict_daily(raw_daily, today)
        print('Prediction horizon        : Daily')
        print(f"Prediction date           : {out['prediction_date'].date()}")
        print(f"Model input date          : {out['input_date'].date()} features (lag-1)")
        print(f"Predicted daily direction : {'UP' if out['pred_label'] == 1 else 'DOWN/FLAT'} ({out['pred_label']})")
        print(f"P(UP)                     : {out['pred_proba_up']:.3f}")
        print(f"Open price                : {out['open_price']:.2f}")
        print(f"Latest price (Close)      : {out['latest_price']:.2f}")
        if out['in_progress']:
            print('Note: For an open market session, "Close" may be intraday last price.')
    except Exception as exc:
        print(f'Daily prediction failed   : {exc}')

print('=' * 70)

Asset                     : S&P 500
Prediction horizon        : Daily
Prediction date           : 2026-04-06
Model input date          : 2026-04-05 features (lag-1)
Predicted daily direction : UP (1)
P(UP)                     : 0.850
Open price                : 6587.66
Latest price (Close)      : 6611.83
Asset                     : Nasdaq
Prediction horizon        : Daily
Prediction date           : 2026-04-06
Model input date          : 2026-04-05 features (lag-1)
Predicted daily direction : UP (1)
P(UP)                     : 0.850
Open price                : 21939.80
Latest price (Close)      : 21996.34
Asset                     : Gold
Prediction horizon        : Daily
Prediction date           : 2026-04-06
Model input date          : 2026-04-05 features (lag-1)
Predicted daily direction : DOWN/FLAT (0)
P(UP)                     : 0.330
Open price                : 4678.60
Latest price (Close)      : 4671.60
Asset                     : Silver
Prediction horizon        : Daily
Predicti

## Weekly Prediction

In [5]:
for asset_name, raw_daily in market_data.items():
    print('=' * 70)
    print(f'Asset                     : {asset_name}')

    try:
        out = predict_weekly(raw_daily, today)
        print('Prediction horizon        : Weekly')
        print(f"Predicted week ending     : {out['prediction_date'].date()}")
        print(f"Model input week ending   : {out['input_date'].date()} features (lag-1)")
        print(f"Predicted weekly direction: {'UP' if out['pred_label'] == 1 else 'DOWN/FLAT'} ({out['pred_label']})")
        print(f"P(UP)                     : {out['pred_proba_up']:.3f}")
        print(f"Week open price           : {out['open_price']:.2f}")
        print(f"Latest weekly close/last  : {out['latest_price']:.2f}")
        if out['in_progress']:
            print('Note: Current week is still in progress until Friday close.')
    except Exception as exc:
        print(f'Weekly prediction failed  : {exc}')

print('=' * 70)

Asset                     : S&P 500
Prediction horizon        : Weekly
Predicted week ending     : 2026-04-10
Model input week ending   : 2026-04-03 features (lag-1)
Predicted weekly direction: UP (1)
P(UP)                     : 0.790
Week open price           : 6587.66
Latest weekly close/last  : 6611.83
Note: Current week is still in progress until Friday close.
Asset                     : Nasdaq
Prediction horizon        : Weekly
Predicted week ending     : 2026-04-10
Model input week ending   : 2026-04-03 features (lag-1)
Predicted weekly direction: UP (1)
P(UP)                     : 0.590
Week open price           : 21939.80
Latest weekly close/last  : 21996.34
Note: Current week is still in progress until Friday close.
Asset                     : Gold
Prediction horizon        : Weekly
Predicted week ending     : 2026-04-10
Model input week ending   : 2026-04-03 features (lag-1)
Predicted weekly direction: UP (1)
P(UP)                     : 0.810
Week open price           : 4678.

## Monthly Prediction

In [ ]:
for asset_name, raw_daily in market_data.items():
    print('=' * 70)
    print(f'Asset                      : {asset_name}')

    try:
        out = predict_monthly(raw_daily, today)
        print('Prediction horizon         : Monthly')
        print(f"Predicted month ending     : {out['prediction_date'].date()}")
        print(f"Model input month ending   : {out['input_date'].date()} features (lag-1)")
        print(f"Predicted monthly direction: {'UP' if out['pred_label'] == 1 else 'DOWN/FLAT'} ({out['pred_label']})")
        print(f"P(UP)                      : {out['pred_proba_up']:.3f}")
        print(f"Month open price           : {out['open_price']:.2f}")
        print(f"Latest monthly close/last  : {out['latest_price']:.2f}")
        if out['in_progress']:
            print('Note: Current month is still in progress until month-end close.')
    except Exception as exc:
        print(f'Monthly prediction failed  : {exc}')

print('=' * 70)

Asset                      : S&P 500
Prediction horizon         : Monthly
Predicted month ending     : 2026-04-30
Model input month ending   : 2026-03-31 features (lag-1)
Predicted monthly direction: DOWN/FLAT (0)
P(UP)                      : 0.205
Month open price           : 6556.56
Latest monthly close/last  : 6611.83
Note: Current month is still in progress until month-end close.
Asset                      : Nasdaq
Prediction horizon         : Monthly
Predicted month ending     : 2026-04-30
Model input month ending   : 2026-03-31 features (lag-1)
Predicted monthly direction: UP (1)
P(UP)                      : 0.770
Month open price           : 21742.79
Latest monthly close/last  : 21996.34
Note: Current month is still in progress until month-end close.
Asset                      : Gold
Prediction horizon         : Monthly
Predicted month ending     : 2026-04-30
Model input month ending   : 2026-03-31 features (lag-1)
Predicted monthly direction: UP (1)
P(UP)                      :

## Quarterly Prediction

In [7]:
for asset_name, raw_daily in market_data.items():
    print('=' * 70)
    print(f'Asset                      : {asset_name}')

    try:
        out = predict_quarterly(raw_daily, today)
        print('Prediction horizon         : Quarterly')
        print(f"Predicted quarter ending   : {out['prediction_date'].date()}")
        print(f"Model input quarter ending : {out['input_date'].date()} features (lag-1)")
        print(f"Predicted quarterly direction: {'UP' if out['pred_label'] == 1 else 'DOWN/FLAT'} ({out['pred_label']})")
        print(f"P(UP)                      : {out['pred_proba_up']:.3f}")
        print(f"Quarter open price         : {out['open_price']:.2f}")
        print(f"Latest quarterly close/last: {out['latest_price']:.2f}")
        if out['in_progress']:
            print('Note: Current quarter is still in progress until quarter-end close.')
    except Exception as exc:
        print(f'Quarterly prediction failed: {exc}')

print('=' * 70)

Asset                      : S&P 500
Prediction horizon         : Quarterly
Predicted quarter ending   : 2026-06-30
Model input quarter ending : 2026-03-31 features (lag-1)
Predicted quarterly direction: UP (1)
P(UP)                      : 0.870
Quarter open price         : 6556.56
Latest quarterly close/last: 6611.83
Note: Current quarter is still in progress until quarter-end close.
Asset                      : Nasdaq
Prediction horizon         : Quarterly
Predicted quarter ending   : 2026-06-30
Model input quarter ending : 2026-03-31 features (lag-1)
Predicted quarterly direction: UP (1)
P(UP)                      : 0.880
Quarter open price         : 21742.79
Latest quarterly close/last: 21996.34
Note: Current quarter is still in progress until quarter-end close.
Asset                      : Gold
Prediction horizon         : Quarterly
Predicted quarter ending   : 2026-06-30
Model input quarter ending : 2026-03-31 features (lag-1)
Predicted quarterly direction: UP (1)
P(UP)          

## Yearly Prediction

In [11]:
for asset_name, raw_daily in market_data.items():
    print('=' * 70)
    print(f'Asset                      : {asset_name}')

    try:
        out = predict_yearly(raw_daily, today)
        print('Prediction horizon         : Yearly')
        print(f"Predicted year ending      : {out['prediction_date'].date()}")
        print(f"Model input year ending    : {out['input_date'].date()} features (lag-1)")
        print(f"Predicted yearly direction : {'UP' if out['pred_label'] == 1 else 'DOWN/FLAT'} ({out['pred_label']})")
        print(f"P(UP)                      : {out['pred_proba_up']:.3f}")
        print(f"Year open price            : {out['open_price']:.2f}")
        print(f"Latest yearly close/last   : {out['latest_price']:.2f}")
        if out['in_progress']:
            print('Note: Current year is still in progress until year-end close.')
    except Exception as exc:
        print(f'Yearly prediction failed   : {exc}')

print('=' * 70)

Asset                      : S&P 500
Prediction horizon         : Yearly
Predicted year ending      : 2026-12-31
Model input year ending    : 2025-12-31 features (lag-1)
Predicted yearly direction : UP (1)
P(UP)                      : 0.645
Year open price            : 6878.11
Latest yearly close/last   : 6611.83
Note: Current year is still in progress until year-end close.
Asset                      : Nasdaq
Prediction horizon         : Yearly
Predicted year ending      : 2026-12-31
Model input year ending    : 2025-12-31 features (lag-1)
Predicted yearly direction : UP (1)
P(UP)                      : 0.540
Year open price            : 23481.49
Latest yearly close/last   : 21996.34
Note: Current year is still in progress until year-end close.
Asset                      : Gold
Prediction horizon         : Yearly
Predicted year ending      : 2026-12-31
Model input year ending    : 2025-12-31 features (lag-1)
Predicted yearly direction : UP (1)
P(UP)                      : 0.565
Year op

## Multi-Horizon Accuracy Backtest

Evaluate walk-forward test accuracy across all time horizons (daily, weekly, monthly, quarterly, yearly).

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


def backtest_horizon_accuracy(raw_daily, horizon='daily', start_date='2023-01-01', min_rows=60):
    """Evaluate walk-forward accuracy for a given time horizon."""
    try:
        if horizon == 'daily':
            data = raw_daily.copy()
            feat = add_intraday_features(data).dropna()
        elif horizon == 'weekly':
            data = raw_daily.resample('W-FRI').agg({
                'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 'Volume': 'sum'
            }).dropna(subset=['Open', 'Close'])
            feat = add_intraday_features(data).dropna()
        elif horizon == 'monthly':
            data = raw_daily.resample('ME').agg({
                'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 'Volume': 'sum'
            }).dropna(subset=['Open', 'Close'])
            feat = add_intraday_features(data).dropna()
        elif horizon == 'quarterly':
            data = raw_daily.resample('QE').agg({
                'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 'Volume': 'sum'
            }).dropna(subset=['Open', 'Close'])
            feat = add_intraday_features(data).dropna()
        elif horizon == 'yearly':
            data = raw_daily.resample('YE').agg({
                'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 'Volume': 'sum'
            }).dropna(subset=['Open', 'Close'])
            feat = add_intraday_features(data).dropna()
        
        eval_idx = feat.index[feat.index >= pd.Timestamp(start_date)]
        y_true, y_pred = [], []
        
        for eval_date in eval_idx:
            hist = feat.loc[:eval_date].copy()
            if len(hist) < min_rows:
                continue
            
            X = hist[INTRADAY_FEATURES]
            y = (hist['Close'] > hist['Open']).astype(int)
            train_mask = X.index < eval_date
            X_train = X.loc[train_mask]
            y_train = y.loc[train_mask]
            
            if len(X_train) < min_rows:
                continue
            
            model = _fit_model_tuned(X_train.values, y_train.values)
            x_now = X.loc[[eval_date]].values
            if x_now.shape[1] != len(INTRADAY_FEATURES):
                continue
            
            y_hat = model.predict(x_now)[0]
            y_actual = y.loc[eval_date]
            y_pred.append(y_hat)
            y_true.append(y_actual)
        
        if len(y_true) < 5:
            return None
        
        return {
            'Horizon': horizon.capitalize(),
            'Accuracy': float(accuracy_score(y_true, y_pred)),
            'Precision': float(precision_score(y_true, y_pred, zero_division=0)),
            'Recall': float(recall_score(y_true, y_pred, zero_division=0)),
            'F1': float(f1_score(y_true, y_pred, zero_division=0)),
            'N_Tests': len(y_true),
        }
    except Exception as e:
        return {'Horizon': horizon.capitalize(), 'Error': str(e)}


results_multi = []
for asset_name in ['S&P 500']:  # Focus on S&P 500
    if asset_name not in market_data:
        continue
    raw = market_data[asset_name]
    for horizon in ['daily', 'weekly', 'monthly', 'quarterly', 'yearly']:
        result = backtest_horizon_accuracy(raw, horizon=horizon, start_date='2023-01-01')
        if result:
            result['Asset'] = asset_name
            results_multi.append(result)

if results_multi:
    df_multi = pd.DataFrame(results_multi)
    print('\nMulti-Horizon Walk-Forward Accuracy (S&P 500)')
    print('=' * 70)
    for col in ['Accuracy', 'Precision', 'Recall', 'F1']:
        if col in df_multi.columns:
            df_multi[col] = df_multi[col].apply(lambda x: f'{x:.4f}' if pd.notna(x) else 'N/A')
    print(df_multi.to_string(index=False))
else:
    print('No multi-horizon backtest results available.')

## Walk-Forward Accuracy Backtest (Daily)

This section evaluates out-of-sample accuracy using a rolling walk-forward setup.
At each test date, the model is trained only on data available up to that date, then predicts that same day's direction.

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import os
import sys
from datetime import datetime, timedelta

# Make this cell runnable standalone (without requiring earlier cells first).
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))
import config


def _local_add_features(df, ma_short, ma_long, vol_window, rsi_window, bb_window):
    d = df.copy()
    d['Direction'] = (d['Close'] > d['Open']).astype(int)
    d['LogReturn'] = np.log(d['Close'] / d['Close'].shift(1))

    d['MA_short'] = d['Close'].rolling(ma_short).mean()
    d['MA_long'] = d['Close'].rolling(ma_long).mean()
    d['MA_cross'] = d['MA_short'] - d['MA_long']

    d['Volatility'] = d['LogReturn'].rolling(vol_window).std()

    delta = d['Close'].diff()
    gain = delta.clip(lower=0).rolling(rsi_window).mean()
    loss = (-delta.clip(upper=0)).rolling(rsi_window).mean()
    rs = gain / (loss + 1e-12)
    d['RSI'] = 100 - (100 / (1 + rs))

    bb_mid = d['Close'].rolling(bb_window).mean()
    bb_std = d['Close'].rolling(bb_window).std()
    d['BB_upper'] = bb_mid + config.BB_STD * bb_std
    d['BB_lower'] = bb_mid - config.BB_STD * bb_std
    d['BB_width'] = (d['BB_upper'] - d['BB_lower']) / bb_mid
    d['BB_pct'] = (d['Close'] - d['BB_lower']) / (d['BB_upper'] - d['BB_lower'])
    return d


def _local_fit_model(X_train, y_train):
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    model = RandomForestClassifier(
        n_estimators=config.N_ESTIMATORS,
        random_state=config.RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(X_train_sc, y_train)
    return model, scaler


def _local_predict_daily(raw_daily, today):
    feat = _local_add_features(
        raw_daily,
        config.MA_SHORT, config.MA_LONG, config.VOL_WINDOW, config.RSI_WINDOW, config.BB_WINDOW,
    )

    features = ['MA_cross', 'Volatility', 'RSI', 'BB_width', 'BB_pct', 'LogReturn']
    X = feat[features].shift(1)
    y = feat['Direction']

    train_mask = X.notna().all(axis=1) & y.notna() & (X.index.date < today.date())
    X_train = X.loc[train_mask]
    y_train = y.loc[train_mask].astype(int)
    if len(X_train) < 200:
        raise ValueError(f'Not enough daily training rows: {len(X_train)}')

    model, scaler = _local_fit_model(X_train, y_train)

    pred_date = today if today in X.index else X.index.max()
    x_now = X.loc[[pred_date]]
    if x_now.isna().any(axis=1).iloc[0]:
        raise ValueError(f'Daily feature row for {pred_date.date()} contains NaN values.')

    x_now_sc = scaler.transform(x_now)
    pred_label = int(model.predict(x_now_sc)[0])

    return {
        'pred_label': pred_label,
    }


# Use notebook predict_daily if already available, otherwise fallback to local one.
predict_daily_fn = predict_daily if 'predict_daily' in globals() else _local_predict_daily


def backtest_daily_walkforward(raw_daily: pd.DataFrame, start_date: str = '2023-01-01', min_train_rows: int = 260) -> dict:
    eval_dates = raw_daily.index[raw_daily.index >= pd.Timestamp(start_date)]
    y_true, y_pred = [], []

    for eval_date in eval_dates:
        hist = raw_daily.loc[:eval_date].copy()
        if len(hist) < min_train_rows:
            continue

        try:
            out = predict_daily_fn(hist, pd.Timestamp(eval_date))
            y_hat = int(out['pred_label'])
            y_actual = int(hist.loc[eval_date, 'Close'] > hist.loc[eval_date, 'Open'])
            y_pred.append(y_hat)
            y_true.append(y_actual)
        except Exception:
            continue

    if len(y_true) == 0:
        raise ValueError('No valid backtest rows. Try an earlier start_date.')

    up_rate = float(np.mean(y_true))
    majority_baseline = max(up_rate, 1.0 - up_rate)

    return {
        'Accuracy': float(accuracy_score(y_true, y_pred)),
        'Balanced Accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'Precision (UP)': float(precision_score(y_true, y_pred, zero_division=0)),
        'Recall (UP)': float(recall_score(y_true, y_pred, zero_division=0)),
        'F1 (UP)': float(f1_score(y_true, y_pred, zero_division=0)),
        'Majority Baseline': majority_baseline,
        'N test bars': int(len(y_true)),
    }


# Auto-load market data if missing.
if 'market_data' not in globals() or not market_data:
    try:
        import yfinance as yf
    except ModuleNotFoundError:
        raise ValueError('yfinance not found. Run the import/setup cell first or install yfinance.')

    today = pd.Timestamp(datetime.now().date())
    fetch_end = (today + timedelta(days=1)).strftime('%Y-%m-%d')

    assets = globals().get('ASSETS', {
        'S&P 500': '^GSPC',
        'Nasdaq': '^IXIC',
        'Gold': 'GC=F',
        'Silver': 'SI=F',
    })

    market_data = {}
    for asset_name, ticker in assets.items():
        raw = yf.download(
            ticker,
            start=config.START_DATE,
            end=fetch_end,
            auto_adjust=True,
            progress=False,
        )
        if isinstance(raw.columns, pd.MultiIndex):
            raw.columns = raw.columns.get_level_values(0)
        if raw.empty:
            continue

        raw = raw.rename_axis('Date').sort_index()
        raw = raw[['Open', 'High', 'Low', 'Close', 'Volume']].dropna(subset=['Open', 'Close'])
        market_data[asset_name] = raw

    if not market_data:
        raise ValueError('No asset data was downloaded from yfinance.')

rows = []
for asset_name, raw_daily in market_data.items():
    try:
        metrics = backtest_daily_walkforward(raw_daily, start_date='2023-01-01')
        rows.append({'Asset': asset_name, **metrics})
    except Exception as exc:
        rows.append({'Asset': asset_name, 'Error': str(exc)})

results_df = pd.DataFrame(rows)

for col in ['Accuracy', 'Balanced Accuracy', 'Precision (UP)', 'Recall (UP)', 'F1 (UP)', 'Majority Baseline']:
    if col in results_df.columns:
        results_df[col] = results_df[col].map(lambda x: f'{x:.3f}' if pd.notna(x) else x)

print('Daily walk-forward backtest summary')
print('=' * 45)
display(results_df)